# Training Baseline Models on Original CXR + COVID-QU-Ex Datasets
Train all 6 architectures on **both** original (non-augmented) datasets.
Each model evaluated on BOTH datasets (cross-dataset generalization).
Optimized for G4 GPU on Google Colab.

## Cell 1 — Install Dependencies

In [1]:
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_LAUNCH_BLOCKING"] = "0"

!pip install -q segmentation-models-pytorch albumentations kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 8.9 MB/s eta 0:00:00


## Cell 2 — Imports & Global Config

In [2]:
import json, random, time, warnings, gc
from pathlib import Path

import numpy as np
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from sklearn.model_selection import train_test_split
import albumentations as A
from albumentations.pytorch import ToTensorV2

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = True

IMG_SIZE     = 256
EPOCHS       = 50
LR           = 5.83e-4
WEIGHT_DECAY = 1e-4
PATIENCE     = 4
MEAN         = (0.485, 0.456, 0.406)
STD          = (0.229, 0.224, 0.225)

BATCH_SIZE   = 128
NUM_WORKERS  = 8
PREFETCH     = 2

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device : cuda
GPU    : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM   : 102.0 GB


## Cell 3 — Google Drive Mount

In [3]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/LungSeg_UNetPP/NB00_baseline")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print(f"Drive dir: {DRIVE_DIR}")

Mounted at /content/drive
Drive dir: /content/drive/MyDrive/LungSeg_UNetPP/NB00_baseline


## Cell 4 — Kaggle Authentication + Download Datasets

In [4]:
from google.colab import files
import os, json

uploaded = files.upload()

os.makedirs("/root/.kaggle", exist_ok=True)
creds = list(uploaded.values())[0]
with open("/root/.kaggle/kaggle.json", "wb") as f:
    f.write(creds)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

creds_dict = json.loads(creds)
os.environ["KAGGLE_USERNAME"]    = creds_dict["username"]
os.environ["KAGGLE_KEY"]         = creds_dict["key"]
os.environ["KAGGLE_CONFIG_DIR"]  = "/root/.kaggle"
print(f"Authenticated as: {creds_dict['username']} ✓")

import kagglehub

print("\nDownloading CXR original dataset...")
CXR_ROOT = Path(kagglehub.dataset_download(
    "felipemeganha/chest-xray-masks-and-labels-images"))
print(f"CXR root     : {CXR_ROOT}")

print("\nDownloading COVID-QU-Ex original dataset...")
COVIDQU_ROOT = Path(kagglehub.dataset_download(
    "anasmohammedtahir/covidqu"))
print(f"COVIDQU root : {COVIDQU_ROOT}")

Saving kaggle.json to kaggle.json
Authenticated as: elhamidyabderahman ✓

Using Colab cache for faster access to the 'chest-xray-masks-and-labels-images' dataset.
CXR root     : /kaggle/input/chest-xray-masks-and-labels-images

Using Colab cache for faster access to the 'covidqu' dataset.
COVIDQU root : /kaggle/input/covidqu


## Cell 5 — Inspect Dataset Structures

In [5]:
print("=== CXR top-level ===")
for item in sorted(CXR_ROOT.iterdir()):
    print(f"  {item.name}")

print("\n=== COVID-QU-Ex top-level ===")
for item in sorted(COVIDQU_ROOT.iterdir()):
    print(f"  {item.name}")

=== CXR top-level ===
  CXR_png
  masks

=== COVID-QU-Ex top-level ===
  COVID-QU-Ex dataset.txt
  Infection Segmentation Data
  Lung Segmentation Data


## Cell 6 — Pair Collection Functions

In [6]:
def collect_cxr_pairs(root: Path):
    """Find CXR image/mask pairs — tries multiple folder name variants."""
    img_candidates  = ["CXR_png", "images", "Images", "img"]
    mask_candidates = ["masks", "masks_png", "Masks", "mask"]
    img_dir = mask_dir = None
    for name in img_candidates:
        if (root / name).exists():
            img_dir = root / name; break
    for name in mask_candidates:
        if (root / name).exists():
            mask_dir = root / name; break
    if img_dir is None or mask_dir is None:
        raise FileNotFoundError(
            f"Could not locate images/masks under {root}\n"
            f"Contents: {[p.name for p in root.iterdir()]}"
        )
    print(f"  CXR images : {img_dir}")
    print(f"  CXR masks  : {mask_dir}")
    pairs = []
    for img_path in sorted(img_dir.glob("*.png")):
        for mask_name in [img_path.stem + "_mask.png", img_path.name]:
            mask_path = mask_dir / mask_name
            if mask_path.exists():
                pairs.append((str(img_path), str(mask_path)))
                break
    print(f"  CXR pairs  : {len(pairs)}")
    return pairs


def collect_covidqu_pairs(root: Path):
    """Find COVID-QU-Ex lung segmentation pairs from original dataset."""
    candidates = [
        root / "Lung Segmentation Data" / "Lung Segmentation Data",
        root / "Lung Segmentation Data",
        root / "lung segmentation data",
    ]
    lung_root = None
    for c in candidates:
        if c.exists():
            lung_root = c; break
    if lung_root is None:
        raise FileNotFoundError(
            f"Could not find Lung Segmentation Data under {root}\n"
            f"Contents: {[p.name for p in root.iterdir()]}"
        )
    print(f"  COVIDQU lung root : {lung_root}")
    pairs = []
    for split in ["Train", "Val", "Test"]:
        split_dir = lung_root / split
        if not split_dir.exists():
            continue
        for cat_dir in sorted(split_dir.iterdir()):
            if not cat_dir.is_dir():
                continue
            img_dir  = cat_dir / "images"
            mask_dir = cat_dir / "lung masks"
            if not mask_dir.exists():
                mask_dir = cat_dir / "masks"
            if not img_dir.exists() or not mask_dir.exists():
                continue
            for img_path in sorted(img_dir.glob("*.png")):
                for mask_name in [img_path.name, img_path.stem + "_mask.png"]:
                    mask_path = mask_dir / mask_name
                    if mask_path.exists():
                        pairs.append((str(img_path), str(mask_path)))
                        break
    print(f"  COVIDQU pairs : {len(pairs)}")
    return pairs


print("Collecting CXR pairs...")
cxr_pairs = collect_cxr_pairs(CXR_ROOT)

print("\nCollecting COVID-QU-Ex pairs...")
covidqu_pairs = collect_covidqu_pairs(COVIDQU_ROOT)

assert len(cxr_pairs)     > 0, "No CXR pairs found!"
assert len(covidqu_pairs) > 0, "No COVID-QU-Ex pairs found!"

  CXR images : /kaggle/input/chest-xray-masks-and-labels-images/CXR_png
  CXR masks  : /kaggle/input/chest-xray-masks-and-labels-images/masks
  CXR pairs  : 703

  COVIDQU lung root : /kaggle/input/covidqu/Lung Segmentation Data/Lung Segmentation Data
  COVIDQU pairs : 33920


## Cell 7 — Train / Val / Test Splits (80 / 10 / 10, seed=42)

In [7]:
def make_splits(pairs, seed=42):
    train, tmp = train_test_split(pairs, test_size=0.20, random_state=seed)
    val, test  = train_test_split(tmp,   test_size=0.50, random_state=seed)
    return train, val, test

cxr_train,     cxr_val,     cxr_test     = make_splits(cxr_pairs)
covidqu_train, covidqu_val, covidqu_test = make_splits(covidqu_pairs)

print(f"CXR         — train: {len(cxr_train):,}  val: {len(cxr_val):,}  test: {len(cxr_test):,}")
print(f"COVID-QU-Ex — train: {len(covidqu_train):,}  val: {len(covidqu_val):,}  test: {len(covidqu_test):,}")

CXR         — train: 562  val: 70  test: 71
COVID-QU-Ex — train: 27,136  val: 3,392  test: 3,392


## Cell 8 — Dataset & Transforms

In [8]:
class SegDataset(Dataset):
    def __init__(self, pairs, transform=None):
        self.pairs     = pairs
        self.transform = transform

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]

        img  = cv2.imread(img_path)
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if img.shape[:2]  != (IMG_SIZE, IMG_SIZE):
            img  = cv2.resize(img,  (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_LINEAR)
        if mask.shape[:2] != (IMG_SIZE, IMG_SIZE):
            mask = cv2.resize(mask, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)

        mask = (mask > 127).astype(np.float32)

        if self.transform:
            aug  = self.transform(image=img, mask=mask)
            img  = aug["image"]
            mask = aug["mask"]

        return img, mask.unsqueeze(0)


train_tf = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                       rotate_limit=15, p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

val_tf = A.Compose([
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

## Cell 9 — Loss & Metrics

In [9]:
bce_loss  = nn.BCEWithLogitsLoss()
dice_loss = smp.losses.DiceLoss(mode="binary")

def combined_loss(pred, target):
    return 0.5 * bce_loss(pred, target) + 0.5 * dice_loss(pred, target)


def compute_metrics(pred_logits, targets, threshold=0.5):
    preds = (torch.sigmoid(pred_logits) > threshold).float()
    tp = (preds * targets).sum()
    fp = (preds * (1 - targets)).sum()
    fn = ((1 - preds) * targets).sum()
    tn = ((1 - preds) * (1 - targets)).sum()
    dice  = (2 * tp) / (2 * tp + fp + fn + 1e-8)
    iou   = tp / (tp + fp + fn + 1e-8)
    pixel = (tp + tn) / (tp + tn + fp + fn + 1e-8)
    return dice.item(), iou.item(), pixel.item()

## Cell 10 — Model Factory

In [10]:
# (arch_key, arch_name, encoder, supports_scse)
ARCHITECTURES = [
    ("unetpp",        "UNet++",     "efficientnet-b4", True),
    ("unetpp",        "UNet++",     "resnet50",        True),
    ("unetpp",        "UNet++",     "resnet34",        True),
    ("unet",          "UNet",       "efficientnet-b4", True),
    ("deeplabv3plus", "DeepLabV3+", "efficientnet-b4", False),
    ("manet",         "MAnet",      "efficientnet-b4", False),
]


def build_model(arch_key, encoder, supports_scse):
    kwargs = dict(encoder_name=encoder, encoder_weights="imagenet",
                  in_channels=3, classes=1)
    if supports_scse:
        kwargs["decoder_attention_type"] = "scse"
    if arch_key == "unetpp":
        return smp.UnetPlusPlus(**kwargs)
    elif arch_key == "unet":
        return smp.Unet(**kwargs)
    elif arch_key == "deeplabv3plus":
        return smp.DeepLabV3Plus(**kwargs)
    elif arch_key == "manet":
        return smp.MAnet(**kwargs)
    else:
        raise ValueError(f"Unknown arch: {arch_key}")


def make_run_id(arch_key, encoder, train_ds):
    enc = encoder.replace("-", "").replace("_", "")
    return f"baseline_{arch_key}_{enc}_{train_ds}"

## Cell 11 — Training & Evaluation Helpers

In [11]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    for imgs, masks in loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        preds = model(imgs)
        loss  = combined_loss(preds, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
    return total_loss / len(loader.dataset)


def validate(model, loader):
    model.eval()
    total_loss, total_dice, total_iou, total_pixel = 0., 0., 0., 0.
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            loss  = combined_loss(preds, masks)
            d, i, p = compute_metrics(preds, masks)
            total_loss  += loss.item() * imgs.size(0)
            total_dice  += d * imgs.size(0)
            total_iou   += i * imgs.size(0)
            total_pixel += p * imgs.size(0)
    n = len(loader.dataset)
    return total_loss/n, total_dice/n, total_iou/n, total_pixel/n


def evaluate_dataset(model, pairs, name):
    loader = DataLoader(
        SegDataset(pairs, val_tf),
        batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=True, prefetch_factor=PREFETCH,
    )
    total_loss, total_dice, total_iou, total_pixel = 0., 0., 0., 0.
    model.eval()
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            preds = model(imgs)
            loss  = combined_loss(preds, masks)
            d, i, p = compute_metrics(preds, masks)
            total_loss  += loss.item() * imgs.size(0)
            total_dice  += d * imgs.size(0)
            total_iou   += i * imgs.size(0)
            total_pixel += p * imgs.size(0)
    n = len(loader.dataset)
    results = {
        "dataset"  : name,
        "n_samples": n,
        "loss"     : round(total_loss  / n, 4),
        "dice"     : round(total_dice  / n, 4),
        "iou"      : round(total_iou   / n, 4),
        "pixel_acc": round(total_pixel / n, 4),
    }
    print(f"\n  {'─'*45}")
    print(f"  Eval on {name} ({n:,} samples)")
    for k, v in results.items():
        if k not in ("dataset", "n_samples"):
            print(f"    {k:12s}: {v:.4f}")
    return results


def save_ood_samples(model, ood_pairs, run_id, ood_name, out_dir):
    samples = ood_pairs[:4]
    fig, axes = plt.subplots(4, 3, figsize=(10, 14))
    model.eval()
    for row, (img_path, mask_path) in enumerate(samples):
        img_rgb = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        mask_gt = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        # resize to IMG_SIZE before inference
        img_resized  = cv2.resize(img_rgb, (IMG_SIZE, IMG_SIZE),
                                  interpolation=cv2.INTER_LINEAR)
        mask_resized = cv2.resize(mask_gt, (IMG_SIZE, IMG_SIZE),
                                  interpolation=cv2.INTER_NEAREST)

        aug = val_tf(image=img_resized,
                     mask=(mask_resized > 127).astype(np.float32))
        inp = aug["image"].unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            pred_mask = (torch.sigmoid(model(inp)) > 0.5).squeeze().cpu().numpy()

        axes[row, 0].imshow(img_resized);            axes[row, 0].set_title("Image"); axes[row, 0].axis("off")
        axes[row, 1].imshow(mask_resized, cmap="gray"); axes[row, 1].set_title("GT");   axes[row, 1].axis("off")
        axes[row, 2].imshow(pred_mask,    cmap="gray"); axes[row, 2].set_title("Pred"); axes[row, 2].axis("off")

    plt.suptitle(f"{run_id} — OOD: {ood_name}", fontsize=11, fontweight="bold")
    plt.tight_layout()
    save_path = out_dir / f"{run_id}_ood_{ood_name}_samples.png"
    plt.savefig(save_path, dpi=100, bbox_inches="tight")
    plt.close()
    return save_path

## Cell 12 — Main Training Loop (Both Datasets)

In [12]:
TRAIN_CONFIGS = [
    ("CXR",     cxr_train,     cxr_val,     cxr_test,
     "COVIDQU", covidqu_test),
    ("COVIDQU", covidqu_train, covidqu_val, covidqu_test,
     "CXR",     cxr_test),
]

all_summaries = []

for arch_key, arch_name, encoder, supports_scse in ARCHITECTURES:
    for (train_ds, tr_pairs, va_pairs, te_pairs,
         ood_ds, ood_pairs) in TRAIN_CONFIGS:

        run_id  = make_run_id(arch_key, encoder, train_ds)
        run_dir = DRIVE_DIR / run_id
        run_dir.mkdir(parents=True, exist_ok=True)

        ckpt_path    = run_dir / f"{run_id}_best.pth"
        history_path = run_dir / f"{run_id}_history.json"

        print(f"\n{'='*65}")
        print(f"  RUN : {run_id}")
        print(f"  Train: {train_ds} ({len(tr_pairs):,} pairs)  |  Batch: {BATCH_SIZE}")
        print(f"{'='*65}")

        # Build model
        model    = build_model(arch_key, encoder, supports_scse).to(DEVICE)
        n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  Params : {n_params:,}")

        # DataLoaders
        train_loader = DataLoader(
            SegDataset(tr_pairs, train_tf),
            batch_size=BATCH_SIZE, shuffle=True,
            num_workers=NUM_WORKERS, pin_memory=True,
            persistent_workers=True, prefetch_factor=PREFETCH,
        )
        val_loader = DataLoader(
            SegDataset(va_pairs, val_tf),
            batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=True,
            persistent_workers=True, prefetch_factor=PREFETCH,
        )

        # Optimizer and scheduler
        optimizer = torch.optim.AdamW(model.parameters(),
                                       lr=LR, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS, eta_min=1e-6)

        # Resume support
        history = {"train_loss": [], "val_loss": [], "val_dice": [],
                   "val_iou": [], "val_pixel_acc": [], "lr": []}
        start_epoch   = 0
        best_val_dice = 0.0
        no_improve    = 0

        if ckpt_path.exists():
            print(f"  Resuming from checkpoint...")
            ckpt = torch.load(ckpt_path, map_location=DEVICE)
            model.load_state_dict(ckpt["model_state"])
            optimizer.load_state_dict(ckpt["optimizer_state"])
            scheduler.load_state_dict(ckpt["scheduler_state"])
            best_val_dice = ckpt["best_val_dice"]
            start_epoch   = ckpt["epoch"] + 1
            no_improve    = ckpt.get("no_improve", 0)
            if history_path.exists():
                with open(history_path) as f:
                    history = json.load(f)
            print(f"  Resumed at epoch {start_epoch}, best dice {best_val_dice:.4f}")
        else:
            print("  Starting fresh training run.")

        # Training loop
        print(f"\n  {'Ep':>4} {'TrLoss':>8} {'VLoss':>8} {'VDice':>7} "
              f"{'VIoU':>7} {'VPixAcc':>8} {'LR':>9} {'Time':>7}")
        print(f"  {'-'*62}")

        for epoch in range(start_epoch, EPOCHS):
            t0 = time.time()

            tr_loss             = train_one_epoch(model, train_loader, optimizer)
            vl_loss, vd, vi, vp = validate(model, val_loader)
            scheduler.step()
            current_lr = scheduler.get_last_lr()[0]

            history["train_loss"].append(tr_loss)
            history["val_loss"].append(vl_loss)
            history["val_dice"].append(vd)
            history["val_iou"].append(vi)
            history["val_pixel_acc"].append(vp)
            history["lr"].append(current_lr)

            # Save history
            with open(history_path, "w") as f:
                json.dump(history, f)

            improved = vd > best_val_dice
            if improved:
                best_val_dice = vd
                no_improve    = 0
                torch.save({
                    "epoch"          : epoch,
                    "model_state"    : model.state_dict(),
                    "optimizer_state": optimizer.state_dict(),
                    "scheduler_state": scheduler.state_dict(),
                    "best_val_dice"  : best_val_dice,
                    "no_improve"     : no_improve,
                }, ckpt_path)
                flag = " ✓"
            else:
                no_improve += 1
                flag = ""

            elapsed = time.time() - t0
            print(f"  {epoch+1:>4} {tr_loss:>8.4f} {vl_loss:>8.4f} {vd:>7.4f} "
                  f"{vi:>7.4f} {vp:>8.4f} {current_lr:>9.2e} {elapsed:>6.0f}s{flag}")

            if no_improve >= PATIENCE:
                print(f"\n  Early stopping at epoch {epoch+1} "
                      f"(no improvement for {PATIENCE} epochs).")
                break

        print(f"\n  Best Val Dice: {best_val_dice:.4f}")


        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt["model_state"])

        indist_results = evaluate_dataset(
            model, te_pairs, f"{train_ds} (in-dist)")

        ood_results = evaluate_dataset(
            model, ood_pairs, f"{ood_ds} (out-of-dist)")

        vis_path = save_ood_samples(
            model, ood_pairs, run_id, ood_ds, run_dir)
        print(f"\n  OOD samples saved → {vis_path}")

        epochs_range = range(1, len(history["val_dice"]) + 1)
        fig, axes    = plt.subplots(1, 3, figsize=(15, 4))
        axes[0].plot(epochs_range, history["train_loss"], label="Train")
        axes[0].plot(epochs_range, history["val_loss"],   label="Val")
        axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
        axes[1].plot(epochs_range, history["val_dice"],  color="green")
        axes[1].set_title("Val Dice"); axes[1].set_xlabel("Epoch")
        axes[2].plot(epochs_range, history["val_iou"],   color="orange")
        axes[2].set_title("Val IoU"); axes[2].set_xlabel("Epoch")
        plt.suptitle(f"Training Curves — {run_id}", fontsize=11, fontweight="bold")
        plt.tight_layout()
        curve_path = run_dir / f"{run_id}_curves.png"
        plt.savefig(curve_path, dpi=100, bbox_inches="tight")
        plt.close()
        print(f"  Curves saved → {curve_path}")


        summary = {
            "run_id"        : run_id,
            "arch"          : arch_name,
            "encoder"       : encoder,
            "train_dataset" : train_ds,
            "img_size"      : IMG_SIZE,
            "batch_size"    : BATCH_SIZE,
            "epochs_trained": len(history["val_dice"]),
            "best_val_dice" : round(best_val_dice, 4),
            "in_dist_eval"  : indist_results,
            "ood_eval"      : ood_results,
        }
        summary_path = run_dir / f"{run_id}_summary.json"
        with open(summary_path, "w") as f:
            json.dump(summary, f, indent=2)
        print(f"  Summary saved → {summary_path}")
        print(json.dumps(summary, indent=2))

        all_summaries.append(summary)

        del model, optimizer, scheduler, train_loader, val_loader
        gc.collect()
        torch.cuda.empty_cache()

print(f"\n\n{'='*65}")
print(f"  ALL RUNS COMPLETE — {len(all_summaries)} models trained")
print(f"{'='*65}")


  RUN : baseline_unetpp_efficientnetb4_CXR
  Train: CXR (562 pairs)  |  Batch: 128


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

  Params : 20,924,428
  Resuming from checkpoint...
  Resumed at epoch 21, best dice 0.9603

    Ep   TrLoss    VLoss   VDice    VIoU  VPixAcc        LR    Time
  --------------------------------------------------------------
    22   0.0636   0.0808  0.9601  0.9233   0.9804  3.47e-04    111s
    23   0.0617   0.0806  0.9596  0.9223   0.9801  3.28e-04     25s
    24   0.0598   0.0795  0.9599  0.9229   0.9802  3.10e-04     25s
    25   0.0578   0.0782  0.9597  0.9225   0.9801  2.92e-04     26s

  Early stopping at epoch 25 (no improvement for 4 epochs).

  Best Val Dice: 0.9603

  ─────────────────────────────────────────────
  Eval on CXR (in-dist) (71 samples)
    loss        : 0.0751
    dice        : 0.9665
    iou         : 0.9351
    pixel_acc   : 0.9832

  ─────────────────────────────────────────────
  Eval on COVIDQU (out-of-dist) (3,392 samples)
    loss        : 0.0884
    dice        : 0.9545
    iou         : 0.9130
    pixel_acc   : 0.9791

  OOD samples saved → /content/d

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

  Params : 51,125,448
  Resuming from checkpoint...
  Resumed at epoch 10, best dice 0.9596

    Ep   TrLoss    VLoss   VDice    VIoU  VPixAcc        LR    Time
  --------------------------------------------------------------
    11   0.1348   0.1360  0.9546  0.9132   0.9773  5.16e-04     92s
    12   0.1242   0.1234  0.9585  0.9203   0.9793  5.04e-04     27s
    13   0.1150   0.1157  0.9579  0.9193   0.9790  4.91e-04     27s
    14   0.1073   0.1087  0.9585  0.9203   0.9795  4.77e-04     29s

  Early stopping at epoch 14 (no improvement for 4 epochs).

  Best Val Dice: 0.9596

  ─────────────────────────────────────────────
  Eval on CXR (in-dist) (71 samples)
    loss        : 0.1367
    dice        : 0.9650
    iou         : 0.9324
    pixel_acc   : 0.9823

  ─────────────────────────────────────────────
  Eval on COVIDQU (out-of-dist) (3,392 samples)
    loss        : 0.1548
    dice        : 0.9486
    iou         : 0.9022
    pixel_acc   : 0.9762

  OOD samples saved → /content/d

config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

  Params : 26,281,332
  Resuming from checkpoint...
  Resumed at epoch 12, best dice 0.9605

    Ep   TrLoss    VLoss   VDice    VIoU  VPixAcc        LR    Time
  --------------------------------------------------------------
    13   0.0817   0.0881  0.9594  0.9220   0.9800  4.91e-04     38s
    14   0.0795   0.0862  0.9582  0.9197   0.9792  4.77e-04     25s
    15   0.0756   0.0822  0.9591  0.9214   0.9797  4.63e-04     24s
    16   0.0716   0.0781  0.9603  0.9237   0.9803  4.48e-04     24s

  Early stopping at epoch 16 (no improvement for 4 epochs).

  Best Val Dice: 0.9605

  ─────────────────────────────────────────────
  Eval on CXR (in-dist) (71 samples)
    loss        : 0.0833
    dice        : 0.9671
    iou         : 0.9363
    pixel_acc   : 0.9835

  ─────────────────────────────────────────────
  Eval on COVIDQU (out-of-dist) (3,392 samples)
    loss        : 0.1151
    dice        : 0.9411
    iou         : 0.8889
    pixel_acc   : 0.9733

  OOD samples saved → /content/d

## Cell 13 — Results Table

In [17]:
rows = []
for s in all_summaries:
    rows.append({
        "arch"         : s["arch"],
        "encoder"      : s["encoder"],
        "train_ds"     : s["train_dataset"],
        "epochs"       : s["epochs_trained"],
        "best_val_dice": s["best_val_dice"],
        "indist_dice"  : s["in_dist_eval"]["dice"],
        "indist_iou"   : s["in_dist_eval"]["iou"],
        "ood_dice"     : s["ood_eval"]["dice"],
        "ood_iou"      : s["ood_eval"]["iou"],
    })

df = pd.DataFrame(rows)
csv_path = DRIVE_DIR / "baseline_both_datasets_results.csv"
df.to_csv(csv_path, index=False)
print(f"CSV saved → {csv_path}\n")
print(df.sort_values(["train_ds", "ood_dice"], ascending=[True, False]).to_string(index=False))

CSV saved → /content/drive/MyDrive/LungSeg_UNetPP/NB00_baseline/baseline_both_datasets_results.csv

      arch         encoder train_ds  epochs  best_val_dice  indist_dice  indist_iou  ood_dice  ood_iou
    UNet++        resnet34  COVIDQU      15         0.9803       0.9800      0.9608    0.9670   0.9361
      UNet efficientnet-b4  COVIDQU      12         0.9801       0.9799      0.9607    0.9666   0.9355
     MAnet efficientnet-b4  COVIDQU      16         0.9802       0.9802      0.9612    0.9664   0.9349
    UNet++ efficientnet-b4  COVIDQU      57         0.9808       0.9806      0.9620    0.9662   0.9345
    UNet++        resnet50  COVIDQU      54         0.9815       0.9812      0.9632    0.9660   0.9342
DeepLabV3+ efficientnet-b4  COVIDQU      13         0.9794       0.9793      0.9594    0.9659   0.9340
    UNet++ efficientnet-b4      CXR      36         0.9603       0.9665      0.9351    0.9545   0.9130
     MAnet efficientnet-b4      CXR      31         0.9607       0.9663     

## Cell 14 — Bar Chart: CXR-trained vs COVIDQU-trained OOD Dice

In [18]:
cxr_df     = df[df["train_ds"] == "CXR"].copy()
covidqu_df = df[df["train_ds"] == "COVIDQU"].copy()

merged = pd.merge(
    cxr_df[["arch","encoder","ood_dice"]].rename(columns={"ood_dice":"ood_cxr"}),
    covidqu_df[["arch","encoder","ood_dice"]].rename(columns={"ood_dice":"ood_covidqu"}),
    on=["arch","encoder"], how="outer"
).fillna(0)

merged["label"] = merged.apply(
    lambda r: f"{r.arch}\n{r.encoder.replace('efficientnet-b4','EffB4').replace('resnet','Res')}",
    axis=1
)

x = np.arange(len(merged)); w = 0.35
fig, ax = plt.subplots(figsize=(max(10, len(merged)*2), 5))
bars1 = ax.bar(x - w/2, merged["ood_cxr"],     w, label="Trained on CXR",     color="#2196F3")
bars2 = ax.bar(x + w/2, merged["ood_covidqu"],  w, label="Trained on COVIDQU", color="#4CAF50")
ax.set_ylabel("OOD Dice")
ax.set_title("Baseline — OOD Dice: CXR-trained vs COVIDQU-trained")
ax.set_xticks(x); ax.set_xticklabels(merged["label"], fontsize=8)
ax.set_ylim(0.80, 1.00); ax.legend(); ax.grid(axis="y", alpha=0.3)
for bar in list(bars1) + list(bars2):
    if bar.get_height() > 0:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=7)
plt.tight_layout()
chart_path = DRIVE_DIR / "baseline_cxr_vs_covidqu_ood_dice.png"
plt.savefig(chart_path, dpi=150); plt.close()
print(f"Chart saved → {chart_path}")

Chart saved → /content/drive/MyDrive/LungSeg_UNetPP/NB00_baseline/baseline_cxr_vs_covidqu_ood_dice.png


## Cell 15 — In-dist vs OOD Dice per Training Dataset

In [19]:
for train_ds_name, color_indist, color_ood in [
    ("CXR",     "#2196F3", "#FF9800"),
    ("COVIDQU", "#4CAF50", "#F44336"),
]:
    sub = df[df["train_ds"] == train_ds_name].sort_values("ood_dice", ascending=False)
    if sub.empty:
        continue
    labels = [
        f"{r.arch}\n{r.encoder.replace('efficientnet-b4','EffB4').replace('resnet','Res')}"
        for _, r in sub.iterrows()
    ]
    x = np.arange(len(sub)); w = 0.35
    fig, ax = plt.subplots(figsize=(max(10, len(sub)*2), 5))
    b1 = ax.bar(x - w/2, sub["indist_dice"], w, label="In-dist Dice", color=color_indist)
    b2 = ax.bar(x + w/2, sub["ood_dice"],    w, label="OOD Dice",     color=color_ood)
    ax.set_ylabel("Dice")
    ax.set_title(f"Baseline {train_ds_name}-trained — In-dist vs OOD Dice")
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8)
    ax.set_ylim(0.80, 1.00); ax.legend(); ax.grid(axis="y", alpha=0.3)
    for bar in list(b1) + list(b2):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f"{bar.get_height():.4f}", ha="center", va="bottom", fontsize=7)
    plt.tight_layout()
    p = DRIVE_DIR / f"baseline_{train_ds_name.lower()}_indist_vs_ood.png"
    plt.savefig(p, dpi=150); plt.close()
    print(f"Chart saved → {p}")

Chart saved → /content/drive/MyDrive/LungSeg_UNetPP/NB00_baseline/baseline_cxr_indist_vs_ood.png
Chart saved → /content/drive/MyDrive/LungSeg_UNetPP/NB00_baseline/baseline_covidqu_indist_vs_ood.png


## Cell 16 — List All Files Saved to Drive

In [20]:
print(f"\nAll files saved to {DRIVE_DIR}:\n")
total_size = 0
for f in sorted(DRIVE_DIR.rglob("*")):
    if f.is_file():
        size_mb    = f.stat().st_size / 1e6
        total_size += size_mb
        print(f"  {str(f.relative_to(DRIVE_DIR)):<75}  ({size_mb:>7.1f} MB)")
print(f"\nTotal size: {total_size:.1f} MB")


All files saved to /content/drive/MyDrive/LungSeg_UNetPP/NB00_baseline:

  baseline_both_datasets_results.csv                                           (    0.0 MB)
  baseline_covidqu_indist_vs_ood.png                                           (    0.1 MB)
  baseline_cxr_indist_vs_ood.png                                               (    0.1 MB)
  baseline_cxr_vs_covidqu_ood_dice.png                                         (    0.1 MB)
  baseline_deeplabv3plus_efficientnetb4_COVIDQU/baseline_deeplabv3plus_efficientnetb4_COVIDQU_best.pth  (  218.2 MB)
  baseline_deeplabv3plus_efficientnetb4_COVIDQU/baseline_deeplabv3plus_efficientnetb4_COVIDQU_curves.png  (    0.1 MB)
  baseline_deeplabv3plus_efficientnetb4_COVIDQU/baseline_deeplabv3plus_efficientnetb4_COVIDQU_history.json  (    0.0 MB)
  baseline_deeplabv3plus_efficientnetb4_COVIDQU/baseline_deeplabv3plus_efficientnetb4_COVIDQU_ood_CXR_samples.png  (    0.4 MB)
  baseline_deeplabv3plus_efficientnetb4_COVIDQU/baseline_deeplabv3plus_ef